# 09 — Retention Recommendation Engine

**Objective:** Map customer segments to personalized CRM campaigns, adjust priorities based on churn probability and FVS, and export recommendations for targeting.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional

from src.config import *
from src.utils import logger, timer, save_csv

## 1. Recommendation Rules

Define retention campaign actions, costs, priorities, and expected retention lifts per micro-segment.

In [2]:
RECOMMENDATION_RULES = {
    "VIP At Risk": {
        "priority": "CRITICAL",
        "actions": [
            "Tier protection guarantee for next 12 months",
            "Personal account manager outreach call within 48 hours",
            "Complimentary airport lounge passes (4 visits)",
            "Bonus 2x miles on next 3 flights",
            "Exclusive partner hotel upgrade offer",
        ],
        "estimated_cost_per_customer": 250,
        "expected_retention_lift": 0.25,
    },
    "Champions": {
        "priority": "LOW",
        "actions": [
            "Exclusive referral program with double rewards",
            "Early access to seasonal promotions",
            "Partner airline upgrade offers",
            "Anniversary milestone celebration email",
            "Invitation to exclusive loyalty events",
        ],
        "estimated_cost_per_customer": 50,
        "expected_retention_lift": 0.05,
    },
    "Loyal Travelers": {
        "priority": "MEDIUM",
        "actions": [
            "Milestone celebration email with bonus points",
            "Cross-sell premium cabin services",
            "Family member enrollment discount (50% off)",
            "Personalized destination recommendations",
            "Credit card co-brand offer with signup bonus",
        ],
        "estimated_cost_per_customer": 75,
        "expected_retention_lift": 0.10,
    },
    "Growth Potential": {
        "priority": "MEDIUM",
        "actions": [
            "Double miles campaign for next 3 months",
            "Destination-based targeted promotions",
            "Credit card co-brand offer with accelerated earning",
            "Weekend getaway package deals",
            "Tier status challenge (earn status faster)",
        ],
        "estimated_cost_per_customer": 100,
        "expected_retention_lift": 0.15,
    },
    "Dormant Members": {
        "priority": "HIGH",
        "actions": [
            "Win-back offer: 50% bonus miles on next booking",
            "'We miss you' personalized email campaign",
            "Limited-time status match offer",
            "Flash sale notification for preferred routes",
            "Points expiry reminder with extension offer",
        ],
        "estimated_cost_per_customer": 150,
        "expected_retention_lift": 0.20,
    },
}

PRIORITY_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}

## 2. Recommendation Logic

Define a function to extract top actions and override priority thresholds for high-risk, high-value members.

In [3]:
def get_recommendation(
    segment: str,
    churn_probability: float = 0.5,
    fvs: float = 50.0,
) -> Dict:
    rule = RECOMMENDATION_RULES.get(segment, RECOMMENDATION_RULES["Dormant Members"])

    priority = rule["priority"]
    if churn_probability > 0.7 and fvs > 60:
        priority = "CRITICAL"
    elif churn_probability > 0.8:
        priority = "HIGH"

    actions = rule["actions"][:3]

    if churn_probability > 0.8:
        urgency = "Immediate action required — high churn probability"
    elif churn_probability > 0.5:
        urgency = "Action recommended within 1 week"
    else:
        urgency = "Scheduled campaign — standard timeline"

    return {
        "segment": segment,
        "priority": priority,
        "priority_order": PRIORITY_ORDER.get(priority, 3),
        "actions": actions,
        "primary_action": actions[0],
        "urgency": urgency,
        "estimated_cost": rule["estimated_cost_per_customer"],
        "expected_retention_lift": rule["expected_retention_lift"],
        "churn_probability": churn_probability,
        "fvs": fvs,
    }

## 3. Batch Process and Export

Generate recommendations for the full customer base, sort by priority, and save the target list for CRM loading.

In [4]:
def batch_recommendations(df: pd.DataFrame) -> pd.DataFrame:
    recommendations = []
    for _, row in df.iterrows():
        segment = row.get("Segment", "Dormant Members")
        churn_prob = row.get("Churn_Probability", row.get("Churn", 0.5))
        fvs = row.get("FVS", 50.0)

        rec = get_recommendation(segment, churn_prob, fvs)
        recommendations.append({
            PK: row[PK],
            "Segment": segment,
            "Churn_Risk": f"{churn_prob * 100:.0f}%",
            "Churn_Probability": churn_prob,
            "FVS": fvs,
            "Priority": rec["priority"],
            "Priority_Order": rec["priority_order"],
            "Primary_Action": rec["primary_action"],
            "All_Actions": " | ".join(rec["actions"]),
            "Urgency": rec["urgency"],
            "Estimated_Cost": rec["estimated_cost"],
            "Expected_Retention_Lift": rec["expected_retention_lift"],
        })

    result = pd.DataFrame(recommendations)
    result = result.sort_values(["Priority_Order", "Churn_Probability"], ascending=[True, False])
    return result

def export_recommendations_csv(recommendations: pd.DataFrame, filepath: Optional[Path] = None):
    if filepath is None:
        filepath = FEATURES_DIR / "retention_recommendations.csv"
    export_cols = [PK, "Segment", "Churn_Risk", "FVS", "Priority",
                   "Primary_Action", "All_Actions", "Urgency", "Estimated_Cost"]
    save_csv(recommendations[export_cols], filepath)

segmented_path = FEATURES_DIR / "customer_segmented.csv"
df = pd.read_csv(segmented_path)
recs = batch_recommendations(df)
export_recommendations_csv(recs)

print(recs["Priority"].value_counts())

2026-06-12 11:28:29 | airline_loyalty | INFO | Saved: retention_recommendations.csv (16,737 rows × 9 cols)


Priority
LOW         6153
MEDIUM      5695
HIGH        2705
CRITICAL    2184
Name: count, dtype: int64
